# Lab 4：MobileNet、ShuffleNet 与 HybridSN

本 Notebook 对应实验四，集中保留可以独立运行的网络定义：MobileNet 的深度可分离卷积、ShuffleNet 的 channel shuffle、SENet/CBAM 注意力，以及 HybridSN 的 3D→2D 主干。

HybridSN 的第一层必须是 `Conv3d(1, 8, ...)`，不能把输出通道误写成 64；后续结构按照 `1 → 8 → 16 → 32` 连接。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, 3, stride, 1, groups=in_channels, bias=False)
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = F.relu(self.depthwise(x))
        return F.relu(self.bn(self.pointwise(x)))

def channel_shuffle(x, groups):
    b, c, h, w = x.shape
    assert c % groups == 0
    x = x.reshape(b, groups, c // groups, h, w)
    x = x.transpose(1, 2).contiguous()
    return x.reshape(b, c, h, w)

class ShuffleBlock(nn.Module):
    def __init__(self, channels, groups=2):
        super().__init__()
        self.group_conv = nn.Conv2d(channels, channels, 1, groups=groups, bias=False)
        self.bn = nn.BatchNorm2d(channels)
        self.dw = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)

    def forward(self, x):
        x = F.relu(self.bn(self.group_conv(x)))
        x = self.dw(x)
        return channel_shuffle(x, 2)

x = torch.randn(2, 3, 32, 32)
print('MobileNet block:', DepthwiseSeparableConv(3, 16)(x).shape)
print('ShuffleNet block:', ShuffleBlock(16)(torch.randn(2, 16, 32, 32)).shape)

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden), nn.ReLU(),
            nn.Linear(hidden, channels), nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        weight = F.adaptive_avg_pool2d(x, 1).reshape(b, c)
        weight = self.fc(weight).reshape(b, c, 1, 1)
        return x * weight

class CBAM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channel = SEBlock(channels)
        self.spatial = nn.Conv2d(2, 1, 7, padding=3)

    def forward(self, x):
        x = self.channel(x)
        avg = x.mean(dim=1, keepdim=True)
        max_value = x.max(dim=1, keepdim=True).values
        attention = torch.sigmoid(self.spatial(torch.cat([avg, max_value], dim=1)))
        return x * attention

feature = torch.randn(2, 32, 16, 16)
print('SE:', SEBlock(32)(feature).shape)
print('CBAM:', CBAM(32)(feature).shape)

In [ ]:
class HybridSN(nn.Module):
    def __init__(self, class_num=16):
        super().__init__()
        self.conv3d = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=(7, 3, 3)),
            nn.ReLU(),
            nn.Conv3d(8, 16, kernel_size=(5, 3, 3)),
            nn.ReLU(),
            nn.Conv3d(16, 32, kernel_size=(3, 3, 3)),
            nn.ReLU()
        )
        self.conv2d = nn.Sequential(
            nn.Conv2d(32 * 18, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.classifier = nn.Linear(64, class_num)

    def forward(self, x):
        x = self.conv3d(x)
        x = x.flatten(start_dim=1, end_dim=2)
        x = self.conv2d(x).flatten(1)
        return self.classifier(x)

model = HybridSN().to(device)
dummy = torch.randn(4, 1, 30, 25, 25, device=device)
logits = model(dummy)
print('HybridSN output:', logits.shape)
loss = F.cross_entropy(logits, torch.randint(0, 16, (4,), device=device))
loss.backward()
print('one-step smoke test passed, loss:', round(loss.item(), 4))

## Indian Pines 真数据接入说明

HybridSN 的真实输入应为 `N×1×30×25×25`：30 是 PCA 后保留的光谱主成分，25×25 是以像素为中心裁出的空间邻域。准备好 `Indian_pines_corrected.mat` 和 `Indian_pines_gt.mat` 后，可以用 `scipy.io.loadmat` 读取，再按照报告中的 PCA、边界裁剪、训练测试划分步骤生成张量。

本 Notebook 的随机输入只用于检查网络维度和反向传播，不代表 Indian Pines 的分类精度；真实精度必须以真实数据、固定随机种子和独立测试流程为准。